## Creating a custom Weather Tool in Langchain: We will create a custom tool call get_weather that return mock weather for city .

#### Key take way : you can plu any safe python function or later your own APIs, so the agent can use your business logic on demand 

In [1]:
# Install packages
!pip install -qU langchain langchain-google-genai google-generativeai==0.8.5



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#  Set your Gemini API key\n AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"
):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")
print(os.environ.get("GOOGLE_API_KEY"))

AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg


In [4]:
from typing import Dict
from datetime import datetime
from zoneinfo import ZoneInfo

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

In [5]:
# Simple mock dataset (extend or swap with your API later)
MOCK_WEATHER: Dict[str, Dict[str, str]] = {
    "Dakar":  {"condition": "Humid, partly cloudy", "temp_c": "31", "humidity": "75%", "rain_chance": "40%"},
    "Saint Louis":   {"condition": "Hot, dry",             "temp_c": "34", "humidity": "35%", "rain_chance": "10%"},
    "Matam":{"condition": "Warm, breezy",        "temp_c": "30", "humidity": "55%", "rain_chance": "20%"},
    "Casamance":{"condition": "Mild, cloudy",        "temp_c": "26", "humidity": "65%", "rain_chance": "30%"},
}

In [6]:
# Help return current timestamp
def _now_ist_str() -> str:
    ist = ZoneInfo("Africa/Dakar")
    return datetime.now(ist).strftime("%Y-%m-%d %H:%M IST")

In [7]:
@tool
def get_weather(city: str) -> str:
    """Return a short, human-friendly weather report for the given city (mock data)."""
    if not city or not city.strip():
        return "Please provide a city name."
    key = city.strip().lower()
    data = MOCK_WEATHER.get(key)
    if not data:
        # Sensible default when city isn't in our mock set
        data = {"condition": "Partly cloudy", "temp_c": "30", "humidity": "60%", "rain_chance": "20%"}
    return (
        f"City: {city.strip().title()}\n"
        f"Time: {_now_ist_str()}\n"
        f"Condition: {data['condition']}\n"
        f"Temperature: {data['temp_c']} °C\n"
        f"Humidity: {data['humidity']}\n"
        f"Chance of rain: {data['rain_chance']}"
    )


In [8]:
def build_weather_agent():
    llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
    tools = [get_weather]

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a helpful weather assistant.\n"
         "RULES:\n"
         "1) Always call the get_weather tool to answer weather questions.\n"
         "2) If the city is missing or unclear, ask the user to specify a city.\n"
         "3) Return concise, friendly answers.\n"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])

    agent = create_tool_calling_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=False)

weather_agent = build_weather_agent()
print(" Weather agent ready.")


 Weather agent ready.


In [9]:
def ask(q: str):
    res = weather_agent.invoke({"input": q})
    print(res["output"], "\n")

# Known city
ask("What’s the weather in Dakar?")

# Another known city
ask("Weather update for Bengaluru, please.")

# Unknown or missing city → agent should ask to clarify
ask("What’s the weather like today?")


c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The weather in Dakar is currently partly cloudy with a temperature of 30°C. Humidity is at 60%, and there is a 20% chance of rain.', 'index': 0, 'extras': {'signature': 'EuUCCuICARFNMg9aCrrGTTVGgdEcR3/LSikCsEGC/gRqqhWrbwey+X09h3a+mkm4Qmsu2W+vXPcKUP6iLruvPwP0FaMwmlOVbM2veN1ijb6IGLYEHBkdEaia13v2OCxebZZTHKbcioOv7a0t+hD+Xnb/qLKmmcDfodR9/zdkJbmGj3oQ2RlO6YO6hA9g+RDn6qvmafQFPwZR9LdQzHE1gEfxiOy2aX2Kik8Ty6bfbErEWMT30+neJ12vc0OsXS63P559t5hChu6UCreMZyj76F09mZEC+a20Ni+dYBy9zE00v/y7VS8kepw/7zGpRZrBxPrWlK2WpihuFPoWXRcjB4DWIN6uUU8zh4Wf4b4I/SdFtiji0bwwkrAORMuTwQlWrWsvERTjWPBEpYDHI8zpG7JVar418kz0Qsc9OHHoQT6TU1/w7gVV1QN3tEOC5diqKBaLQf67WjD17BanZudpzxqc5m9k3zAX'}}] 



c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The weather in Bengaluru is currently **partly cloudy** with a temperature of **30 °C**. Humidity is at 60%, and there is a 20% chance of rain. Have a wonderful day!', 'index': 0, 'extras': {'signature': 'EsgBCsUBARFNMg/WdjKsTRIXzEL9T88HVQIJqpqVCmhhk4UrUjFF1v9rYs8/Qrd7867vISJI+JXRi5YSsETcOc31jkCbEwwOFMGLLMMbSXrl044iG50OczWbiJuZQZ5hvMH0/hfFqmbtTGsHmPJdZFdxJHqdYVClyASxgCpSBlmTZ6Eo13ecHGcV5wRNaJp8plX0p0q04d04DF8kz00uoV5Ph3+nNteHM3bqIe0Zd4TAZJJ+3Irj9q0KFoAPrs1bMjWEAFhmYkBPOr8='}}] 



c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Which city would you like to know the weather for?', 'index': 0, 'extras': {'signature': 'EoADCv0CARFNMg9PDsj7eW20HREKjf9ILQj9s0kEO23W7OQoxO43pRcSgbjWv/rVE21kBsTzhaNTg63oNB6W61oCEPDbQnZ/6aY+bs3AdpAkIiFBSTZ8UtDwNpWh8Kp6C/ON+Cys8B/RUf9h1LLgsasAU+/iD6tCQ5hZtWgRPyQW5E4DCQge9xtvdXUHpmhKZbZD1+3e/F44c15E+4tTp4mWESqlg89pcNuV6Z7kfGtuuiq8kjP4wrhxZOmlvp5N3LSaJL/qSuuwvJq58OStCzju7ltSIpJxX2p2GFnF74eXnfjITlZgW1UXkkA9/b/lfRpjlFsiVsL32rE1aa65CZ1aQhtbsa7EVN9ombJ3a4uUeCXIIsIzmaE56OpV1uu4FPyBrPAsP5WMM6q4lPci6fl1dJZ4DQx3xDy4BHcyqN8dSqVOnQarksOREiw80aCsy/6vaPetm+YpOaiX8yMNoPIN3jRrM8/wlFF/UBKHXFZTUBKUgoG8qJmEk8YtKsGlkKmj'}}] 

